[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soleildayana/Oumuamua-Project/blob/main/from_scratch.ipynb)

# OUMUAMUA - En búsqueda de su estrella madre

## Librerías

In [1]:
!pip install pymcel -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.7 MB/s eta 0:00:00


In [2]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import spiceypy as spy

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!



## Estado inicial de referencia (base común)
Extraer un estado heliocéntrico de 1I/'Oumuamua cercano a perihelio y utilizarlo como condición de referencia para todos los experimentos. Según wikipedia, esto representa la época del 14 de octubre de 2017.

In [3]:
epoch_ref = '2017-10-14'  # Fecha de referencia cercana al perihelio

def extrae_estado(resultado):
    if isinstance(resultado, tuple) and len(resultado) >= 3:
        estado = np.asarray(resultado[2], dtype=float)
        if estado.ndim == 2:
            estado = estado[0]
        return estado
    return np.asarray(resultado, dtype=float)

raw = pc.consulta_horizons(
    id='1I',
    location='@0',
    epochs=epoch_ref,
    datos='vectors',
    propiedades=[('x', 'km'), ('y', 'km'), ('z', 'km'), ('vx', 'km/s'), ('vy', 'km/s'), ('vz', 'km/s')]
)

estado_ref = extrae_estado(raw)
r0, v0 = estado_ref[:3], estado_ref[3:]

print('Estado extraído correctamente:')
print('r0 [km]:', r0)
print('v0 [km/s]:', v0)


Estado extraído correctamente:
r0 [km]: [ 1.45267492e+08  7.47620204e+07 -1.07128187e+07]
v0 [km/s]: [44.83537051 10.39892688 14.33813591]


## Caracterización de la hipérbola
Se obtienen elementos orbitales con `estado_a_elementos(mu, estado)` y se valida energía específica positiva, mostrando que el objeto no pertenece a nuestro sistema solar.

In [5]:
mu_sun = pc.constantes.mu_sun  # valor base en SI (m^3/s^2)
mu_sun_km = mu_sun / 1e9           # conversión a km^3/s^2 (consistente con r0, v0)

print('Parámetro gravitacional del Sol [m^3/s^2]:', mu_sun)
print('Parámetro gravitacional del Sol [km^3/s^2]:', mu_sun_km)

# elementos = (p, e, i, Omega, omega, f) según convención de pymcel.
p, e, inc, Omega, omega, f = pc.estado_a_elementos(mu_sun_km, estado_ref)

print('Elementos orbitales extraídos correctamente:')
print('p [km]:', p)
print('e:', e)
print('i [°]:', np.degrees(inc))
print('Omega [°]:', np.degrees(Omega))
print('omega [°]:', np.degrees(omega))
print('f [°]:', np.degrees(f))

Parámetro gravitacional del Sol [m^3/s^2]: 1.3271244004127942e+20
Parámetro gravitacional del Sol [km^3/s^2]: 132712440041.27942
Elementos orbitales extraídos correctamente:
p [km]: 85604594.26472798
e: 1.20554107345583
i [°]: 123.1137532641553
Omega [°]: 24.781488427458203
omega [°]: 242.20374636009197
f [°]: 113.31585543976549


In [6]:
# Validación física rápida para clasificar la cónica
ere = 0.5 * np.dot(v0, v0) - mu_sun_km / np.linalg.norm(r0)
mare = np.linalg.norm(np.cross(r0, v0))


print('Energía específica [km^2/s^2]:', ere)
print('¿Órbita hiperbólica (epsilon>0: )?', ere > 0)
print('Momento angular específico [km^2/s]:', mare)

Energía específica [km^2/s^2]: 351.3972315370671
¿Órbita hiperbólica (epsilon>0: )? True
Momento angular específico [km^2/s]: 3370577781.8670444


## Geometría Analítica

"mirar hacia atrás" en el tiempo.
Método Matemático: Utilizando la excentricidad (e) obtenida, calcularás el ángulo de las asíntotas de la hipérbola mediante la relación cosψ=1/e
.
Rutina auxiliar: Implementarás las matrices de rotación de Euler (M(ω,i,Ω)) para orientar el vector de la asíntota de entrada en el sistema de referencia inercial. Esto dirá exactamente desde qué dirección del espacio profundo llegó el objeto.

In [7]:
# Geometría hiperbólica + orientación inercial con Euler M(ω, i, Ω)

# Ángulo de asíntota (según clase): cos(psi)=1/e
psi = np.arccos(1.0 / e)

# Anomalía verdadera al infinito: cos(f_inf) = -1/e
f_inf = np.arccos(-1.0 / e)

def R1(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0],
                     [0, c, -s],
                     [0, s,  c]])

def R3(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[ c, -s, 0],
                     [ s,  c, 0],
                     [ 0,  0, 1]])

# Matriz de rotación perifocal -> inercial
M_euler = R3(Omega) @ R1(inc) @ R3(omega)

# Direcciones asintóticas en marco perifocal (posición)
rhat_in_pf  = np.array([np.cos(-f_inf), np.sin(-f_inf), 0.0])  # entrada (t -> -inf)
rhat_out_pf = np.array([np.cos(+f_inf), np.sin(+f_inf), 0.0])  # salida  (t -> +inf)

# Dirección de velocidad asintótica en perifocal
vhat_in_pf = np.array([-np.sin(-f_inf), e + np.cos(-f_inf), 0.0])
vhat_in_pf = vhat_in_pf / np.linalg.norm(vhat_in_pf)

# Transformación a marco inercial heliocéntrico
rhat_in_ine  = M_euler @ rhat_in_pf
rhat_out_ine = M_euler @ rhat_out_pf
vhat_in_ine  = M_euler @ vhat_in_pf

# Dirección "desde dónde llegó": opuesta a la velocidad de entrada
u_from = -vhat_in_ine / np.linalg.norm(vhat_in_ine)

def ra_dec(u):
    ux, uy, uz = u / np.linalg.norm(u)
    ra = np.degrees(np.arctan2(uy, ux)) % 360.0
    dec = np.degrees(np.arcsin(uz))
    return ra, dec

ra_from, dec_from = ra_dec(u_from)

print(f"e = {e:.9f}")
print(f"psi (cos psi = 1/e) = {np.degrees(psi):.6f} deg")
print(f"f_inf = {np.degrees(f_inf):.6f} deg")
print("\nVector asintótico de entrada:", rhat_in_ine)
print("Vector asintótico de salida:", rhat_out_ine)
print("Dirección de llegada desde espacio profundo u_from:", u_from)
print(f"Radiant de llegada (RA, Dec) = ({ra_from:.6f} deg, {dec_from:.6f} deg)")

e = 1.205541073
psi (cos psi = 1/e) = 33.952277 deg
f_inf = 146.047723 deg

Vector asintótico de entrada: [ 0.13030606 -0.53808449  0.83275771]
Vector asintótico de salida: [0.90815065 0.13445231 0.39646561]
Dirección de llegada desde espacio profundo u_from: [ 0.13030606 -0.53808449  0.83275771]
Radiant de llegada (RA, Dec) = (283.613049 deg, 56.383073 deg)


## Visualización de la Trayectoria

In [8]:
# -- Rutina para generar la órbita con transformaciones --

def orbital_elements_to_orbit_3d(e, a, i, node, peri, num_points=100):
    """
    Convierte los elementos orbitales clásicos en una órbita 3D en el espacio.

    Args:
        e (float): Excentricidad.
        a (float): Semieje mayor.
        i (float): Inclinación (en radianes).
        node (float): Longitud del nodo ascendente (en radianes).
        peri (float): Argumento del periapsis (en radianes).
        num_points (int): Número de puntos para generar la órbita.

    Returns:
        np.ndarray: Array de Nx3 con las coordenadas (x, y, z) de la órbita.
    """

    # Calcular el parámetro semilato recto (p)
    # En hipérbola, 'a' suele ser negativa; usamos |a| para mantener p>0 en r = p/(1+e cos f).
    p = a * (1 - e**2) if e < 1 else abs(a) * (e**2 - 1) # Asegurar que 'p' es positivo para hipérbolas
    if e == 1: # Caso parabólico
        p = 2 * a # a es la distancia al periapsis para una parábola

    # Generar ángulos de anomalía verdadera (f) para dibujar la órbita
    # Si es una órbita hiperbólica, los límites de f son diferentes.
    if e < 1: # Elipse
        fs = np.linspace(0, 2 * np.pi, num_points)
    elif e == 1: # Parábola, se necesita un rango adecuado
        # Para una parábola, la anomalía verdadera puede ir de -pi a pi, pero la órbita es infinita.
        # Usaremos un rango que muestre una buena parte de la trayectoria.
        f_limit = np.arccos(-1/e + 0.001) # Límite para evitar division por cero
        fs = np.linspace(-f_limit, f_limit, num_points) # Valores cercanos al periapsis
    else: # Hipérbola
        # Los límites de la anomalía verdadera para una hipérbola
        f_limit = np.arccos(-1/e + 0.001) # Pequeña perturbación para evitar arccos(-1)
        fs = np.linspace(-f_limit + 0.1, f_limit - 0.1, num_points) # Evitar los asíntotas

    # Calcular la distancia radial (r) para cada f
    rs = p / (1 + e * np.cos(fs))

    # Coordenadas en el plano perifocal (xy) con z=0
    xfs = rs * np.cos(fs)
    yfs = rs * np.sin(fs)
    zfs = np.zeros_like(xfs)

    # Convertir a array de 3xN para multiplicación matricial
    r_perifocal = np.array([xfs, yfs, zfs])

    # Matrices de rotación (del sistema perifocal al astronómico)
    # Nota: spiceypy.rotate(angle, axis) devuelve la matriz de rotación CONTRARIA
    # a la que se usa para transformar un vector, si se usa como M @ v.
    # Para pasar del sistema perifocal al astronómico, las rotaciones son las inversas.
    # O lo que es lo mismo, rotar por ángulos negativos en el orden inverso.

    # Rotaciones de Euler (Z-X-Z)
    # 1. Rotación alrededor del eje Z por -Omega (Longitud del Nodo Ascendente)
    Rz_minus_Omega = spy.rotate(-node, 3)
    # 2. Rotación alrededor del eje X por -i (Inclinación)
    Rx_minus_i = spy.rotate(-i, 1)
    # 3. Rotación alrededor del eje Z por -w (Argumento del Periapsis)
    Rz_minus_peri = spy.rotate(-peri, 3)

    # Combinar las matrices para obtener la transformación de perifocal a astronómico
    M_perifocal2astro = Rz_minus_Omega @ Rx_minus_i @ Rz_minus_peri

    # Aplicar la transformación
    r_astro = M_perifocal2astro @ r_perifocal

    # Transponer para obtener un array de Nx3 (N puntos, 3 coordenadas)
    return r_astro.T

In [9]:
# --Rutina para generar el visualizador --

def create_asteroid_viewer(a, e, i, node, peri, name="Asteroide", grid_lim=2.5):
    """
    Genera y muestra un visualizador 3D para un cuerpo menor.
    """
    # Elementos orbitales de la Tierra (J2000)
    deg = np.pi/180
    a_earth = 1.0000010178
    e_earth = 0.0167086
    i_earth = 0.00005 * deg
    node_earth = -11.26064 * deg
    peri_earth = 102.94719 * deg

    # 1. Generar trayectorias
    orbit_body = orbital_elements_to_orbit_3d(e=e, a=a, i=i, node=node, peri=peri, num_points=600)
    orbit_earth = orbital_elements_to_orbit_3d(e=e_earth, a=a_earth, i=i_earth, node=node_earth, peri=peri_earth, num_points=400)

    fig = go.Figure()

    # --- Sol ---
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(size=8, color='yellow', symbol='circle'), name='Sol'))

    # --- Dirección Vernal (X) ---
    fig.add_trace(go.Scatter3d(x=[0, grid_lim*0.5], y=[0, 0], z=[0, 0],
                               mode='lines', line=dict(color='magenta', width=2), name='Eje X (Vernal)'))

    # --- Órbita de la Tierra ---
    fig.add_trace(go.Scatter3d(x=orbit_earth[:, 0], y=orbit_earth[:, 1], z=orbit_earth[:, 2],
                               mode='lines', line=dict(color='#40a0ff', width=2, dash='dot'), name='Órbita Tierra'))

    # --- Órbita del Objeto ---
    fig.add_trace(go.Scatter3d(x=orbit_body[:, 0], y=orbit_body[:, 1], z=orbit_body[:, 2],
                               mode='lines', line=dict(color='white', width=4), name=f'Órbita {name}'))

    # --- Picket-fence (Líneas de inclinación) ---
    for p in orbit_body[::5]:
        fig.add_trace(go.Scatter3d(
            x=[p[0], p[0]], y=[p[1], p[1]], z=[0, p[2]],
            mode='lines',
            line=dict(color='rgba(200, 200, 200, 0.4)', width=2),
            showlegend=False
        ))

    # --- Periapsis Marker ---
    peri_pos = orbit_body[0] if e < 1 else orbit_body[len(orbit_body)//2] # Aproximación visual
    fig.add_trace(go.Scatter3d(x=[orbit_body[0,0]], y=[orbit_body[0,1]], z=[orbit_body[0,2]],
                               mode='markers+text', text=['Peri'], textposition='top center',
                               marker=dict(color='red', size=5), name='Periapsis'))

    # --- Línea de Nodos ---
    node_vec = np.array([np.cos(node), np.sin(node), 0]) * (grid_lim * 0.7)
    fig.add_trace(go.Scatter3d(
        x=[-node_vec[0], node_vec[0]], y=[-node_vec[1], node_vec[1]], z=[0, 0],
        mode='lines', line=dict(color='#00ffff', width=3, dash='longdash'), name='Línea de Nodos'
    ))

    # --- Plano Eclíptico ---
    grid_r = np.linspace(-grid_lim, grid_lim, 20)
    xx, yy = np.meshgrid(grid_r, grid_r)
    fig.add_trace(go.Surface(x=xx, y=yy, z=np.zeros_like(xx),
                             opacity=0.1, showscale=False, colorscale=[[0, 'cyan'], [1, 'cyan']], hoverinfo='skip'))

    fig.update_layout(
        title=f"Simulación de Órbita: {name}",
        scene=dict(
            xaxis=dict(title='X (AU)', range=[-grid_lim, grid_lim]),
            yaxis=dict(title='Y (AU)', range=[-grid_lim, grid_lim]),
            zaxis=dict(title='Z (AU)', range=[-grid_lim, grid_lim]),
            aspectmode='cube',
            bgcolor='black',
            camera=dict(eye=dict(x=1.8, y=1.2, z=1.2))
        ),
        template="plotly_dark",
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

In [10]:
a_km = p/(1-e**2)
# pc.constantes.au está en metros; al dividir por 1000 obtenemos km por AU.
KM_PER_AU = pc.constantes.au / 1000
a_au = a_km / KM_PER_AU
create_asteroid_viewer(a=a_au, e=e, i=inc, node=Omega, peri=omega, name="1I/'Oumuamua")
